# Mobile Sales Data Analysis
**Author:** Vahid Hushen  
**Program:** AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026 | BharatCares

## Project objective
Analyze mobile-phone sales records to understand sales value, units sold, transaction patterns, brand/model performance, city distribution, payment methods, and customer ratings.

**Important:** Sales value is calculated from `Units Sold × Price Per Unit` when a pre-calculated sales column is not present. All numeric findings and charts are computed from the dataset at run time.


## 1. Import libraries and load the dataset

In [ ]:
from pathlib import Path
import urllib.request
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

DATA_FILE = Path("Mobile Sales Data (2).xlsx")
DATA_URL = "https://raw.githubusercontent.com/DataWithRohit/Mobile-Sales-Dashboard/main/Mobile%20Sales%20Data%20(2).xlsx"

if not DATA_FILE.exists():
    try:
        print("Downloading the public dataset...")
        urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    except Exception as exc:
        raise RuntimeError(
            "Dataset download failed. Open the source repository listed in README.md, "
            "download 'Mobile Sales Data (2).xlsx', and place it in this notebook's folder. "
            f"Details: {exc}"
        )

# Load the first worksheet and normalize column labels.
raw = pd.read_excel(DATA_FILE)
df = raw.copy()
df.columns = [
    re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower())).strip("_")
    for c in df.columns
]
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")
display(df.head())
print("Columns:", df.columns.tolist())


## 2. Data quality checks

In [ ]:
initial_rows = len(df)
duplicate_rows = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

missing_report = df.isna().sum().sort_values(ascending=False).rename("missing_values").to_frame()
print(f"Exact duplicate rows removed: {initial_rows - len(df)}")
display(missing_report)
display(df.dtypes.astype(str).rename("data_type").to_frame())


## 3. Identify fields and prepare numeric columns

In [ ]:
def find_column(*aliases):
    # Match normalized column names exactly first.
    for alias in aliases:
        key = re.sub(r"[^a-z0-9]+", "_", alias.lower()).strip("_")
        if key in df.columns:
            return key
    # Then allow partial matches for common variations.
    for alias in aliases:
        key = re.sub(r"[^a-z0-9]+", "_", alias.lower()).strip("_")
        for col in df.columns:
            if key and (key in col or col in key):
                return col
    return None

date_col = find_column("date", "transaction date", "sales date")
brand_col = find_column("brand", "mobile brand")
model_col = find_column("mobile model", "model", "phone model")
units_col = find_column("units sold", "quantity sold", "quantity", "units")
price_col = find_column("price per unit", "unit price", "price")
sales_col = find_column("total sales", "sales amount", "revenue", "sales")
transaction_col = find_column("transaction id", "transactions")
city_col = find_column("city", "location")
payment_col = find_column("payment method", "payment type", "payment")
rating_col = find_column("customer ratings", "customer rating", "rating")
year_col = find_column("year")
month_col = find_column("month")
day_name_col = find_column("day name", "weekday")

field_map = pd.DataFrame({
    "Analysis field": ["Date", "Brand", "Mobile model", "Units sold", "Price per unit",
                       "Sales value", "Transaction ID/count", "City", "Payment method",
                       "Customer rating", "Year", "Month", "Day name"],
    "Detected column": [date_col, brand_col, model_col, units_col, price_col, sales_col,
                        transaction_col, city_col, payment_col, rating_col, year_col,
                        month_col, day_name_col]
})
display(field_map)

for col in [units_col, price_col, sales_col, rating_col, year_col]:
    if col:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(r"[,₹$]", "", regex=True).str.strip(),
                                errors="coerce")

# In this dataset, total sales is commonly derived from units sold multiplied by price per unit.
if sales_col is None and units_col and price_col:
    df["calculated_sales"] = df[units_col] * df[price_col]
    sales_col = "calculated_sales"
    print("Created calculated_sales = units sold × price per unit.")

if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    # If the source contains separate Year/Month/Day fields instead of a date field, create a date.
elif year_col and month_col and day_name_col is None:
    print("No single date column detected; time-trend analysis may be limited.")


## 4. Key performance indicators

In [ ]:
kpis = {
    "Rows after duplicate removal": int(len(df)),
    "Columns": int(df.shape[1]),
    "Duplicate rows remaining": int(df.duplicated().sum())
}
if sales_col:
    kpis["Total sales value"] = float(pd.to_numeric(df[sales_col], errors="coerce").sum())
    kpis["Average sales value per record"] = float(pd.to_numeric(df[sales_col], errors="coerce").mean())
if units_col:
    kpis["Total units sold"] = float(pd.to_numeric(df[units_col], errors="coerce").sum())
if transaction_col:
    kpis["Unique transaction IDs"] = int(df[transaction_col].nunique())
else:
    kpis["Number of records (transaction proxy)"] = int(len(df))
if date_col and df[date_col].notna().any():
    kpis["First date"] = str(df[date_col].min().date())
    kpis["Last date"] = str(df[date_col].max().date())

display(pd.DataFrame({"KPI": list(kpis.keys()), "Value": list(kpis.values())}))


## 5. Sales trend by month

In [ ]:
if date_col and sales_col:
    trend = df[[date_col, sales_col]].copy()
    trend[sales_col] = pd.to_numeric(trend[sales_col], errors="coerce")
    trend = trend.dropna()
    if not trend.empty:
        trend["month_period"] = trend[date_col].dt.to_period("M").dt.to_timestamp()
        monthly_sales = trend.groupby("month_period", as_index=False)[sales_col].sum()
        display(monthly_sales)
        plt.figure(figsize=(11, 5))
        plt.plot(monthly_sales["month_period"], monthly_sales[sales_col], marker="o")
        plt.title("Monthly Mobile Sales Value")
        plt.xlabel("Month")
        plt.ylabel("Sales value")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("No valid rows available for monthly trend.")
else:
    print("Skipped: a date column and sales value are needed for monthly trend.")


## 6. Brand and model analysis

In [ ]:
def category_analysis(category_col, metric_col, title, top_n=10):
    if not category_col or not metric_col:
        print(f"Skipped '{title}': required columns were not detected.")
        return None
    temp = df[[category_col, metric_col]].copy()
    temp[metric_col] = pd.to_numeric(temp[metric_col], errors="coerce")
    temp = temp.dropna(subset=[category_col, metric_col])
    if temp.empty:
        print(f"Skipped '{title}': no usable records.")
        return None
    result = temp.groupby(category_col)[metric_col].sum().sort_values(ascending=False).head(top_n)
    display(result.rename("total").to_frame())
    plt.figure(figsize=(9, 5))
    result.sort_values().plot(kind="barh")
    plt.title(title)
    plt.xlabel(metric_col.replace("_", " ").title())
    plt.ylabel(category_col.replace("_", " ").title())
    plt.tight_layout()
    plt.show()
    return result

category_analysis(brand_col, sales_col, "Sales Value by Brand")
category_analysis(model_col, sales_col, "Sales Value by Mobile Model")
category_analysis(city_col, sales_col, "Sales Value by City")


## 7. Units sold by brand and model

In [ ]:
category_analysis(brand_col, units_col, "Units Sold by Brand")
category_analysis(model_col, units_col, "Units Sold by Mobile Model")


## 8. Payment methods and customer ratings

In [ ]:
if payment_col:
    print("Records by payment method:")
    display(df[payment_col].value_counts(dropna=False).rename("record_count").to_frame())
    if sales_col:
        category_analysis(payment_col, sales_col, "Sales Value by Payment Method")
else:
    print("Payment-method field not found.")

if rating_col:
    print("Customer rating counts:")
    ratings = df[rating_col].value_counts(dropna=False).sort_index()
    display(ratings.rename("record_count").to_frame())
    plt.figure(figsize=(8, 4))
    ratings.plot(kind="bar")
    plt.title("Customer Rating Distribution")
    plt.xlabel("Customer rating")
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.show()
else:
    print("Customer-rating field not found.")


## 9. Conclusions and limitations

In [ ]:
print("Analysis run completed.")
print(f"Final dataset size: {df.shape[0]:,} rows × {df.shape[1]} columns.")
print("Use the actual KPI table and charts above to write dataset-specific findings.")
print("Limitations: results depend on data quality and the fields available in the source workbook.")
print("This project performs descriptive analysis; it does not train a machine-learning model.")


## Reproducibility note
Run all cells from top to bottom after placing the dataset workbook in the same folder as this notebook. Save the notebook after execution if you want to retain the output tables and charts. Review the generated outputs before stating specific business findings in the report.
